In [ ]:
!pip install roboflow

In [ ]:
!pip install roboflow

import os
from roboflow import Roboflow

# Set ROBOFLOW_API_KEY as an environment variable or Kaggle Secret.
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("kaushiks-workspace-6ofyp").project("weapon-detection-dinou-qdunh-r0nrb-l6ap8")
version = project.version(1)
dataset = version.download("yolov5")
                

In [ ]:
import os

dataset_path = dataset.location

for root, dirs, files in os.walk(dataset_path):
    print(root)
    break

In [ ]:
import os

dataset_path = "/kaggle/working/Weapon-Detection-1"

for item in os.listdir(dataset_path):
    print(item)

In [ ]:
import yaml
import os

dataset_path = "/kaggle/working/Weapon-Detection-1"
yaml_path = os.path.join(dataset_path, "data.yaml")

# Load YAML
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Fix paths (absolute paths)
data['train'] = os.path.join(dataset_path, "train/images")
data['val'] = os.path.join(dataset_path, "valid/images")  # important
data['test'] = os.path.join(dataset_path, "test/images")

# Save YAML
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("YAML FIXED SUCCESSFULLY ✅")
print(data)

In [ ]:
import os

train_images_path = dataset_path + "/train/images"
train_labels_path = dataset_path + "/train/labels"

images = os.listdir(train_images_path)
labels = os.listdir(train_labels_path)

print("Total Images:", len(images))
print("Total Labels:", len(labels))

In [ ]:
missing = []

for img in images:
    label = img.replace(".jpg", ".txt").replace(".png", ".txt")
    if label not in labels:
        missing.append(img)

print("Missing labels:", len(missing))

In [ ]:
import os

dataset_path = "/kaggle/working/Weapon-Detection-1"

def count_images(folder):
    return len(os.listdir(os.path.join(dataset_path, folder, "images")))

print("Train Images:", count_images("train"))
print("Valid Images:", count_images("valid"))
print("Test Images:", count_images("test"))

In [ ]:
import yaml

with open(dataset_path + "/data.yaml", 'r') as f:
    data = yaml.safe_load(f)

class_names = data['names']
print("Classes:", class_names)

In [ ]:
from collections import defaultdict

label_path = os.path.join(dataset_path, "train/labels")

class_counts = defaultdict(int)

for file in os.listdir(label_path):
    with open(os.path.join(label_path, file), 'r') as f:
        for line in f:
            cls = int(line.split()[0])
            class_counts[cls] += 1

# Print results
for k, v in class_counts.items():
    print(f"{class_names[k]}: {v}")

In [ ]:
import matplotlib.pyplot as plt

names = [class_names[k] for k in class_counts.keys()]
values = [class_counts[k] for k in class_counts.keys()]

plt.figure(figsize=(10,5))
plt.bar(names, values)
plt.xticks(rotation=45)
plt.title("Class Distribution (Train Set)")
plt.xlabel("Classes")
plt.ylabel("Number of Objects")
plt.show()

YOLO

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov10s.pt")

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO("yolov10s.pt")

# Fine-tune on your dataset
results = model.train(
    data="/kaggle/working/Weapon-Detection-1/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,  # GPU

    # 🔥 Fine-tuning friendly settings
    lr0=0.001,   # lower learning rate (important)
    
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,

    name="yolov10_finetune",
    project="/kaggle/working/weapon_project"
)

In [ ]:
from ultralytics import YOLO

# Load your trained model
model_path = "/kaggle/working/weapon_project/yolov10_finetune/weights/best.pt"
model = YOLO(model_path)

# Run validation
metrics = model.val()

In [ ]:
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map50_95 = metrics.box.map

# Compute F1
f1 = 2 * (precision * recall) / (precision + recall + 1e-6)

print("\n===== EVALUATION METRICS =====")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1 Score      : {f1:.4f}")
print(f"mAP@50        : {map50:.4f}")
print(f"mAP@50-95     : {map50_95:.4f}")
print("==============================")

In [ ]:
print("\nClass-wise mAP50:")
for i, name in enumerate(model.names.values()):
    print(f"{name}: {metrics.box.ap50[i]:.4f}")

In [ ]:
from ultralytics import YOLO

# Load trained model
model = YOLO("/kaggle/working/weapon_project/yolov10_finetune/weights/best.pt")

# Run validation
metrics = model.val()

# Extract precision & recall
precision = metrics.box.mp
recall = metrics.box.mr

# Compute F1 score
f1 = 2 * (precision * recall) / (precision + recall + 1e-6)

# Print
print(f"F1 Score: {f1:.4f}")

In [ ]:
from ultralytics import YOLO

# Load trained model
model = YOLO("/kaggle/working/weapon_project/yolov10_finetune/weights/best.pt")

# Run prediction on test dataset
results = model.predict(
    source="/kaggle/working/Weapon-Detection-1/test/images",
    conf=0.6,          # 🔥 your requirement
    save=True,         # save output images
    save_txt=True      # save predictions as labels (optional)
)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

pred_path = "/kaggle/working/runs/detect/predict"

files = os.listdir(pred_path)[:5]

for f in files:
    img = cv2.imread(os.path.join(pred_path, f))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.imshow(img)
    plt.title(f)
    plt.axis("off")
    plt.show()

In [ ]:
from IPython.display import Image, display

display(Image("/kaggle/working/weapon_project/yolov10_finetune/results.png"))

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Load trained model
model = YOLO("/kaggle/working/weapon_project/yolov10_finetune/weights/best.pt")

# Run validation
metrics = model.val()

# Extract values directly
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50

# Compute F1
f1 = 2 * (precision * recall) / (precision + recall + 1e-6)

# Plot graph
names = ["Precision", "Recall", "F1", "mAP@50"]
values = [precision, recall, f1, map50]

plt.figure()
plt.bar(names, values)
plt.title("YOLOv10 Model Performance (Real Values)")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.show()

In [ ]:
# Get class names
class_names = list(model.names.values())

# Get class-wise mAP@50
class_map = metrics.box.ap50

plt.figure()
plt.bar(class_names, class_map)
plt.xticks(rotation=45)
plt.title("Class-wise mAP@50 (YOLOv10)")
plt.ylabel("mAP")
plt.show()

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from IPython.display import Image, display
import os

# -------------------------------
# Load trained model
# -------------------------------
model_path = "/kaggle/working/weapon_project/yolov10_finetune/weights/best.pt"
model = YOLO(model_path)

# -------------------------------
# Run validation (generates PR + confusion matrix)
# -------------------------------
metrics = model.val(plots=True)

# -------------------------------
# Define paths
# -------------------------------
base_path = "/kaggle/working/weapon_project/yolov10_finetune"

pr_curve_path = os.path.join(base_path, "PR_curve.png")
conf_matrix_path = os.path.join(base_path, "confusion_matrix.png")
results_path = os.path.join(base_path, "results.png")

# -------------------------------
# Display PR Curve
# -------------------------------
print("\n📈 Precision-Recall Curve")
if os.path.exists(pr_curve_path):
    display(Image(pr_curve_path))
else:
    print("PR curve not found")

# -------------------------------
# Display Confusion Matrix
# -------------------------------
print("\n📊 Confusion Matrix")
if os.path.exists(conf_matrix_path):
    display(Image(conf_matrix_path))
else:
    print("Confusion matrix not found")

# -------------------------------
# Display Loss Curves (Training Graph)
# -------------------------------
print("\n📉 Loss Curves & Training Metrics")
if os.path.exists(results_path):
    display(Image(results_path))
else:
    print("Training results graph not found")

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

# Load model
model = YOLO("/kaggle/working/weapon_project/yolov10_finetune/weights/best.pt")

# Run validation again (force plot generation)
metrics = model.val(plots=True)

# Folder where results are saved
base_path = "/kaggle/working/weapon_project/yolov10_finetune"

# List all files
print("\nFiles in directory:")
files = os.listdir(base_path)
for f in files:
    print(f)

# -------------------------------
# Auto display graphs
# -------------------------------
print("\n📊 Displaying Available Graphs:\n")

for f in files:
    if f.endswith(".png"):
        print(f"Showing: {f}")
        display(Image(os.path.join(base_path, f)))

FAST RCNN

In [ ]:
!pip install torchvision

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F

import os
import cv2
import numpy as np

In [ ]:
class WeaponDataset(torch.utils.data.Dataset):
    def __init__(self, root):
        self.root = root
        self.imgs = sorted(os.listdir(os.path.join(root, "images")))
        self.labels = sorted(os.listdir(os.path.join(root, "labels")))

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, "images", self.imgs[idx])
        label_path = os.path.join(self.root, "labels", self.labels[idx])

        # Load image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape

        boxes = []
        labels = []

        # Read YOLO label format
        with open(label_path) as f:
            for line in f:
                cls, x, y, bw, bh = map(float, line.split())

                # Convert YOLO → Pascal VOC format
                xmin = (x - bw/2) * w
                ymin = (y - bh/2) * h
                xmax = (x + bw/2) * w
                ymax = (y + bh/2) * h

                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(int(cls) + 1)  # +1 (background = 0)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}

        img = F.to_tensor(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

In [ ]:
train_dataset = WeaponDataset("/kaggle/working/Weapon-Detection-1/train")
val_dataset = WeaponDataset("/kaggle/working/Weapon-Detection-1/valid")

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x))
)

val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x))
)

In [ ]:
model = fasterrcnn_resnet50_fpn(pretrained=True)

# Modify classifier for 6 classes + background
num_classes = 6 + 1
in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
    in_features, num_classes
)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

In [ ]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

In [ ]:
import time

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_start = time.time()
    
    total_loss = 0
    
    print(f"\n===== Epoch {epoch+1}/{num_epochs} =====")

    for i, (images, targets) in enumerate(train_loader):
        batch_start = time.time()

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        batch_time = time.time() - batch_start
        loss_value = losses.item()
        total_loss += loss_value

        # 🔥 Batch-wise logging
        print(f"Batch {i+1}/{len(train_loader)} | "
              f"Loss: {loss_value:.4f} | "
              f"Time: {batch_time:.2f}s")

    # 🔥 Epoch summary
    epoch_time = time.time() - epoch_start
    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Epoch Time : {epoch_time:.2f}s")
    print("="*40)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for images, targets in val_loader:
        images = [img.to(device) for img in images]
        
        outputs = model(images)

        for i in range(len(outputs)):
            preds = outputs[i]['labels'].cpu().numpy()
            trues = targets[i]['labels'].cpu().numpy()
            
            # handle empty cases
            if len(preds) == 0 or len(trues) == 0:
                continue
            
            min_len = min(len(preds), len(trues))
            
            all_preds.extend(preds[:min_len])
            all_targets.extend(trues[:min_len])

# Compute metrics
precision = precision_score(all_targets, all_preds, average='macro')
recall = recall_score(all_targets, all_preds, average='macro')
f1 = f1_score(all_targets, all_preds, average='macro')

print("\n===== FASTER R-CNN METRICS =====")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print("================================")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os
import random

# -------------------------------
# Config
# -------------------------------
test_path = "/kaggle/working/Weapon-Detection-1/test/images"
class_names = ['Grenade', 'Gun', 'Handgun', 'Knife', 'Pistol', 'Rifle']

model.eval()

# -------------------------------
# Pick random test images
# -------------------------------
images_list = os.listdir(test_path)
sample_images = random.sample(images_list, 5)

# -------------------------------
# Run inference + display
# -------------------------------
for img_name in sample_images:
    img_path = os.path.join(test_path, img_name)

    # Load image
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Convert to tensor
    img_tensor = torch.from_numpy(img_rgb / 255.).permute(2,0,1).float().to(device)

    # Predict
    with torch.no_grad():
        outputs = model([img_tensor])[0]

    boxes = outputs['boxes'].cpu().numpy()
    scores = outputs['scores'].cpu().numpy()
    labels = outputs['labels'].cpu().numpy()

    # -------------------------------
    # Draw boxes (confidence >= 0.6)
    # -------------------------------
    for box, score, label in zip(boxes, scores, labels):
        if score >= 0.6:
            x1, y1, x2, y2 = map(int, box)

            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0,255,0), 2)
            text = f"{class_names[label-1]} {score:.2f}"
            cv2.putText(img_rgb, text, (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    # -------------------------------
    # Show image
    # -------------------------------
    plt.figure(figsize=(6,6))
    plt.imshow(img_rgb)
    plt.title(img_name)
    plt.axis("off")
    plt.show()

YOLO vs Faster R-CNN

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

# -------------------------------
# 🔹 YOLO METRICS (AUTO)
# -------------------------------
yolo_model = YOLO("/kaggle/working/weapon_project/yolov10_finetune/weights/best.pt")

yolo_metrics = yolo_model.val()

yolo_precision = yolo_metrics.box.mp
yolo_recall = yolo_metrics.box.mr

# Compute F1
yolo_f1 = 2 * (yolo_precision * yolo_recall) / (yolo_precision + yolo_recall + 1e-6)

# -------------------------------
# 🔹 Faster R-CNN METRICS (AUTO)
# -------------------------------
# From your evaluation step
frcnn_precision = precision
frcnn_recall = recall
frcnn_f1 = f1

# -------------------------------
# Metrics
# -------------------------------
metrics = ["Precision", "Recall", "F1"]

yolo_values = [yolo_precision, yolo_recall, yolo_f1]
frcnn_values = [frcnn_precision, frcnn_recall, frcnn_f1]

x = np.arange(len(metrics))
width = 0.35

# -------------------------------
# Plot
# -------------------------------
plt.figure()

plt.bar(x - width/2, yolo_values, width, label="YOLOv10")
plt.bar(x + width/2, frcnn_values, width, label="Faster R-CNN")

plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("YOLO vs Faster R-CNN (Precision / Recall / F1)")
plt.ylim(0, 1)
plt.legend()

plt.show()